# 01 — Exploratory Data Analysis
**Star Wars Character Network Analysis**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from collections import Counter

plt.style.use('dark_background')
%matplotlib inline
print('✅ Ready')

## 1. Load Data

In [ ]:
from src.graph_builder import load_processed, build_graph, graph_summary

nodes, edges = load_processed()
G = build_graph(nodes, edges)

print(f'Characters : {len(nodes)}')
print(f'Interactions: {len(edges)}')
nodes.head()

## 2. Graph Summary

In [ ]:
summary = graph_summary(G)
for k, v in summary.items():
    print(f'{k:<40} {v}')

## 3. Degree Distribution

In [ ]:
degrees = [d for _, d in G.degree()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(degrees, bins=20, color='#58a6ff', edgecolor='#0d1117')
axes[0].axvline(np.mean(degrees), color='#ff7b72', linestyle='--',
                label=f'Mean={np.mean(degrees):.1f}')
axes[0].set_title('Degree Distribution')
axes[0].legend()

from collections import Counter
dc = Counter(degrees)
axes[1].scatter(sorted(dc), [dc[x] for x in sorted(dc)], color='#58a6ff')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_title('Log-Log (Power Law Check)')
plt.tight_layout(); plt.show()

## 4. Top Characters by Degree

In [ ]:
top20 = nodes.nlargest(20, 'degree')[['name','degree','total_weight']]
fig = px.bar(top20, x='degree', y='name', orientation='h',
             color='degree', color_continuous_scale='Plasma',
             title='Top 20 Characters by Degree', template='plotly_dark')
fig.update_layout(yaxis=dict(autorange='reversed'), height=550)
fig.show()

## 5. Edge Weight Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].hist(edges['weight'], bins=30, color='#d2a8ff',
             edgecolor='#0d1117')
axes[0].set_title('Raw Edge Weights')
axes[1].hist(edges['weight_norm'], bins=30, color='#3fb950',
             edgecolor='#0d1117')
axes[1].set_title('Normalized Edge Weights')
plt.tight_layout(); plt.show()
print(edges['weight'].describe())

## 6. Clustering Coefficient Distribution

In [ ]:
cc = nx.clustering(G, weight='weight')
plt.figure(figsize=(10, 4))
plt.hist(list(cc.values()), bins=20, color='#ffa657',
         edgecolor='#0d1117')
plt.title('Clustering Coefficient Distribution')
plt.xlabel('Clustering Coefficient'); plt.ylabel('Count')
plt.show()

cc_df = pd.DataFrame([{'name': G.nodes[n]['name'], 'cc': v}
                       for n, v in cc.items()])
print(cc_df.nlargest(10,'cc').to_string(index=False))